# MultiBench Part 3: MOSI — L2-MCTN (Cyclic Translation)

Parts 1–2 assumed every modality is present at test time. **MCTN** targets the case where a modality may be missing or unreliable when you deploy.

**What you'll learn**
- Learning a **joint representation by translating** from a source modality to a target and **cycling back** (there and back).
- Why cyclic translation makes the representation robust — at test time you only need the *source* modality.
- What **"Level 2"** (hierarchical, two translation stages) adds, and what each loss term (`criterion_t0/c/t1/r`, weighted by `mu_*`) does.

Paper: Pham et al., *Found in Translation* (AAAI 2019). Dataset: CMU-MOSI sentiment regression (same as Part 1).

Designed to run on **Google Colab**.

Requirements:
- Free space on Google Drive to clone MultiBench and save results (~2GB).
- Copy this notebook to your own Drive first (File -> "Save a copy in Drive") so your results and trained models persist.

In [ ]:
from google.colab import drive
drive.mount("/content/drive/")

## Clone or update MultiBench

In [ ]:
import os

repo_dir = "/content/drive/MyDrive/multibench"

if os.path.isdir(os.path.join(repo_dir, ".git")):
    print("Repo already exists, pulling latest changes...")
    !git -C "$repo_dir" fetch origin
    !git -C "$repo_dir" reset --hard origin/main
else:
    print("Cloning MultiBench...")
    !git clone https://github.com/human-ai-lab/multibench.git "$repo_dir"

print(f"Repo location: {repo_dir}")

## Setup Python path

In [ ]:
import sys

if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

print(f"Using MultiBench from: {repo_dir}")

## Check and download MOSI data

Data is stored at `MultiBench/data/affect/mosi_raw.pkl` — inside the repo on Drive, so it persists across sessions and matches the path used by the local `.py` scripts.

In [ ]:
data_dir  = os.path.join(repo_dir, "data", "affect")
data_path = os.path.join(data_dir, "mosi_raw.pkl")

os.makedirs(data_dir, exist_ok=True)

if os.path.exists(data_path):
    print(f"Data found: {data_path}")
else:
    print("Downloading mosi_raw.pkl (one-time)...")
    !wget -q --show-progress -O "$data_path" https://filedn.eu/lDTxyzlMbdMJJq0AvECx20X/mosi_raw.pkl
    print(f"Saved to: {data_path}")

print(f"data_path = {data_path}")

## Install dependencies

In [ ]:
!pip install -q memory-profiler

## Setup device

In [ ]:
import torch
from torch import nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Load MOSI dataset

In [ ]:
from datasets.affect.get_data import get_dataloader

traindata, validdata, testdata = get_dataloader(data_path, robust_test=False)

## Define encoder and decoder modules

In [ ]:
from unimodals.common_models import GRU, MLP
from fusions.MCTN import Encoder, Decoder

feature_dim = 300   # width of a raw modality feature vector
hidden_dim = 32     # width of the shared/joint representation

# TRANSLATION STAGE 0: source modality <-> first target modality.
#   encoder0 maps the source into the shared/joint space (hidden_dim);
#   decoder0 maps it back out to a modality (feature_dim) — the 'there and back'.
encoder0 = Encoder(feature_dim, hidden_dim, n_layers=1, dropout=0.0).to(device)
decoder0 = Decoder(hidden_dim, feature_dim, n_layers=1, dropout=0.0).to(device)

# TRANSLATION STAGE 1 (the 'Level 2' part): translate again to fold in the
# next modality, building a richer joint representation.
encoder1 = Encoder(hidden_dim, hidden_dim, n_layers=1, dropout=0.0).to(device)
decoder1 = Decoder(hidden_dim, feature_dim, n_layers=1, dropout=0.0).to(device)

# Summarises the final joint representation into a vector for the head.
reg_encoder = nn.GRU(hidden_dim, 32).to(device)

## Define prediction head

In [ ]:
head = MLP(32, 64, 1).to(device)

## Train L2-MCTN

In [ ]:
from private_test_scripts.all_in_one import all_in_one_train
from training_structures.MCTN_Level2 import train, test

allmodules = [encoder0, decoder0, encoder1, decoder1, reg_encoder, head]

def trainprocess():
    train(
        traindata,
        validdata,
        encoder0,                  # stage-0 encoder (source -> joint)
        decoder0,                  # stage-0 decoder (joint -> target, the cycle)
        encoder1,                  # stage-1 encoder ('Level 2' translation)
        decoder1,                  # stage-1 decoder
        reg_encoder,               # summarises joint repr for the head
        head,                      # joint repr -> sentiment score
        # Loss terms: one per translation/cycle step + the final prediction
        criterion_t0=nn.MSELoss(), # stage-0 translation reconstruction loss
        criterion_c=nn.MSELoss(),  # cyclic-consistency loss (back to source)
        criterion_t1=nn.MSELoss(), # stage-1 translation reconstruction loss
        criterion_r=nn.L1Loss(),   # regression loss on the sentiment score
        max_seq_len=20,
        # mu_* weight each translation loss relative to the regression loss
        mu_t0=0.01, mu_c=0.01, mu_t1=0.01,
        dropout_p=0.15, early_stop=False, patience_num=15,
        lr=1e-4, weight_decay=0.01, op_type=torch.optim.AdamW,
        epoch=50, model_save="results/models/best_mctn.pt")

all_in_one_train(trainprocess, allmodules)

## Test

In [ ]:
model = torch.load("results/models/best_mctn.pt", map_location=device, weights_only=False).to(device)
test(model, testdata, "mosi", no_robust=True)

That concludes Part 3! Feel free to open an issue on GitHub if you have any questions.